# MAKER：唾液/血液检测、联合依赖和两阶段学习
复现 Yang & Xu（2025）的疾病诊断例子，第 2、5、6 节。
[论文 PDF](https://personalpages.manchester.ac.uk/staff/jian-bo.yang/JB%20Yang%20Journal_Papers/YangXu-AI-MAKER.pdf)，DOI 10.1016/j.artint.2025.104289。
6300 条聚合记录；训练参数 21 个；两目标按优先级求解，delta=10⁻⁵。
焦元 H1、H2、Theta={H1,H2} 的模糊质量，与 2^Theta 上的不可信质量分开存储。
Table 18 **不含 prior**，训练目标 **包含 prior**。不能把两者混为同一对照。
源码：[计数/参数](../paper_cases/diagnosis.json)、[组合公式](../../brb/maker/combination.py)、[训练目标](../../brb/maker/objectives.py)。

## 1. 运行环境

In [1]:
from pathlib import Path
import sys, platform
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'brb' / 'rimer').is_dir())
sys.path.insert(0, str(root))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
np.random.seed(2026)
pd.set_option('display.precision', 6)
print('Python:', platform.python_version())
print('Toolkit root located.')


Python: 3.12.14
Toolkit root located.


## 2. 加载聚合计数
训练组是互不重叠的单独唾液、单独血液和联合检测记录；获取证据时使用相应可用的全部记录。

In [2]:
from examples.paper_cases.diagnosis import build_diagnosis_case,apply_paper_parameters
from brb.maker import dependency_indices,MAKERTrainer,paper_objectives
model,data,case=build_diagnosis_case()
display(pd.DataFrame({'group':['saliva only','blood only','both'],'records':[g.counts.sum() for g in data.groups]}))
assert data.total==6300
print('Focal sets:',model.space.labels)

,group,records
0,saliva only,2100.0
1,blood only,3100.0
2,both,1100.0


Focal sets: ('{H1}', '{H2}', '{H1,H2}')


## 3. 按论文获得似然证据
先按每个状态总数计算输入的条件频率，再对该输入的状态似然归一化；不等同于行计数的普通分类概率。

In [3]:
for key in [('saliva',),('blood',),('saliva','blood')]:
    table=model.tables[key]
    print(key)
    display(pd.DataFrame(table.probabilities,index=table.values,columns=['H1','H2','Theta']))

('saliva',)


,H1,H2,Theta
positive,0.627035,0.053463,0.319503
negative,0.044800,0.608279,0.346921


('blood',)


,H1,H2,Theta
positive,0.630511,0.033337,0.336151
negative,0.041097,0.628341,0.330562


('saliva', 'blood')


H1        H2     Theta
positive positive  0.672830  0.016077  0.311093
         negative  0.236842  0.526316  0.236842
negative positive  0.433735  0.240964  0.325301
         negative  0.006939  0.632228  0.360833

## 4. 联合依赖指数
本案例是同一观测状态对应的对角联合支持。一般数据必须提供完整联合支持或前缀表。

In [4]:
a=model.tables[('saliva',)].probabilities[0];b=model.tables[('blood',)].probabilities[0]
joint=model.tables[('saliva','blood')].joint_support[0]
alpha=dependency_indices(a,b,joint)
display(pd.DataFrame(alpha,index=['H1','H2','Theta'],columns=['H1','H2','Theta']))
np.testing.assert_allclose(alpha[0,0],1.7018,atol=5e-5)

,H1,H2,Theta
H1,1.701848,0.000000,0.000000
H2,0.000000,9.020419,0.000000
Theta,0.000000,0.000000,2.896551


## 5. 固定论文参数复现 Table 18
使用 Tables 16–17 的四位小数参数，排除 prior；预期误差在 1.5×10⁻⁴ 内。

In [5]:
paper_model,_,_=build_diagnosis_case();apply_paper_parameters(paper_model,case)
paper_runs=[paper_model.infer(v,include_prior=False) for v in case['joint_order']]
computed=np.array([r.probabilities for r in paper_runs]);paper_reference=np.array(case['paper_reference']['table18'])
comparison=pd.DataFrame({'input':[str(v) for v in case['joint_order']], 'computed_H1':computed[:,0],'paper_reference_H1':paper_reference[:,0],
    'computed_H2':computed[:,1],'paper_reference_H2':paper_reference[:,1],'computed_Theta':computed[:,2],'paper_reference_Theta':paper_reference[:,2]})
display(comparison)
np.testing.assert_allclose(computed,paper_reference,atol=1.5e-4,rtol=0)

,input,computed_H1,paper_reference_H1,computed_H2,paper_reference_H2,computed_Theta,paper_reference_Theta
0,"['positive', 'positive']",0.670643,0.6706,0.022129,0.0221,0.307227,0.3072
1,"['positive', 'negative']",0.264720,0.2647,0.490845,0.4909,0.244435,0.2444
2,"['negative', 'positive']",0.444663,0.4447,0.241543,0.2415,0.313795,0.3138
3,"['negative', 'negative']",0.017797,0.0178,0.644470,0.6445,0.337733,0.3377


## 6. 不可信质量与含 prior 的推理
Table 18 的可信度对照，以及加上先验之后的另一组输出。

In [6]:
display(pd.DataFrame({'computed_trust':[r.trustworthiness for r in paper_runs], 'paper_reference_trust':case['paper_reference']['trustworthiness'],
    'ambiguity':[r.ambiguity for r in paper_runs]},index=[str(v) for v in case['joint_order']]))
np.testing.assert_allclose([r.trustworthiness for r in paper_runs],case['paper_reference']['trustworthiness'],atol=1.5e-4)
display(pd.DataFrame([paper_model.infer(v).probabilities for v in case['joint_order']],index=[str(v) for v in case['joint_order']],columns=['H1 with prior','H2 with prior','Theta with prior']))

,computed_trust,paper_reference_trust,ambiguity
"['positive', 'positive']",0.988245,0.9882,0.307227
"['positive', 'negative']",0.983425,0.9834,0.244435
"['negative', 'positive']",0.982852,0.9829,0.313795
"['negative', 'negative']",0.976888,0.9769,0.337733


,H1 with prior,H2 with prior,Theta with prior
"['positive', 'positive']",0.366951,0.545243,0.087806
"['positive', 'negative']",0.107873,0.847306,0.044822
"['negative', 'positive']",0.199308,0.732959,0.067732
"['negative', 'negative']",0.037438,0.912582,0.049980


## 7. 两阶段优先级 minimax 训练
初值使用论文上界，参数下界 0.5。第一阶段优化观测误差 f1；第二阶段优化参考参数偏离 f2，并约束 f1≤阶段一最优值+10⁻⁵。

In [7]:
fit=MAKERTrainer(maxiter=500,ftol=1e-10,priority_tolerance=1e-5).fit(model,data,bounds=data.bounds)
display(pd.DataFrame(fit.stage_results))
assert fit.success,fit.status
assert len(fit.stage_results)==2
assert fit.stage_results[1]['f1']<=fit.stage_results[0]['f1']+1e-5+1e-8
assert fit.constraint_violation<1e-7

,stage,success,status,f1,f2,iterations,constraint_violation,priority_limit
0,1,True,Optimization terminated successfully,0.007403,0.084729,51,7.974637e-13,NaN
1,2,True,Optimization terminated successfully,0.007413,0.022220,4,5.297349e-11,0.007413


## 8. 训练结果和论文报告值
论文使用 Excel Evolutionary；这里是局部 SLSQP。还列出将论文四位小数参数重新代入 Eq.86–95 的结果。**推理表复现通过，但论文参数与报告的训练目标存在未解释差异**，不能宣称完全复现其训练数值。

In [8]:
trained=np.array([x.max() for x in paper_objectives(model,data)])
recomputed_paper=np.array([x.max() for x in paper_objectives(paper_model,data)])
paper_reference=np.array([case['paper_reference']['f1'],case['paper_reference']['f2']])
display(pd.DataFrame({'computed_SLSQP':trained,'paper_reference_reported':paper_reference,'computed_from_paper_parameters':recomputed_paper},index=['f1','f2']))
print('Difference requires original supplementary Excel to resolve; no objective rescaling was introduced.')

,computed_SLSQP,paper_reference_reported,computed_from_paper_parameters
f1,0.007413,0.0075,0.013553
f2,0.022220,0.0247,0.006200


Difference requires original supplementary Excel to resolve; no objective rescaling was introduced.


## 9. 已学习参数摘要

In [9]:
rows=[]
for key,p in model.parameters.items():
    for kind,values in p.items(): rows.append({'group':str(key),'parameter':kind,'values':np.round(values,6).tolist()})
display(pd.DataFrame(rows))
assert sum(a.size for p in model.parameters.values() for a in p.values())==21

,group,parameter,values
0,"('saliva',)",weights,"[0.899956, 0.796997, 0.502081]"
1,"('saliva',)",reliabilities,"[0.899122, 0.9]"
2,"('blood',)",weights,"[0.95, 0.725387, 0.5]"
3,"('blood',)",reliabilities,"[0.95, 0.95]"
4,"('saliva', 'blood')",weights,"[0.85, 0.85, 0.85]"
5,"('saliva', 'blood')",reliabilities,"[0.85, 0.85, 0.85, 0.85]"
6,"('prior',)",weights,"[1.0, 0.5, 0.5]"
7,"('prior',)",reliabilities,[0.898471]


## 10. 验收与适用范围
验证了本案例的证据获取、依赖指数、固定参数推理及可行的两阶段学习。集合输出默认保留；需要单一分类时必须显式选择决策变换。

In [10]:
assert fit.objective_after<fit.objective_before
for v in case['joint_order']:
    r=model.infer(v)
    np.testing.assert_allclose(r.erps_mass.sum()+r.untrustworthiness,1,atol=1e-12)
print('Verified: Table18, trustworthiness, 21 parameters, priority-constrained learning.')

Verified: Table18, trustworthiness, 21 parameters, priority-constrained learning.
